# 噪声测试

> 测试环境噪声以及action噪声对训练好的模型的影响

## 环境噪声测试

环境噪声指的是观测不准，不应该修改原始的meta_env文件，而是在 evaluate_policy 函数中，对 s_ 进行进一步的修改，切记使用深拷贝，以免影响到env.Res

In [7]:
import numpy as np
from ppo_discrete import PPO_discrete
from meta_env import EpidemicModel
import pandas as pd
import warnings
from train import add_noise_to_state

warnings.filterwarnings("ignore")

from config import args
from config import experiment_scenario



In [8]:
import contextlib

@contextlib.contextmanager
def set_args_temporarily(args, **kwargs):
    original_values = {key: getattr(args, key) for key in kwargs}
    for key, value in kwargs.items():
        setattr(args, key, value)
    try:
        yield
    finally:
        for key, value in original_values.items():
            setattr(args, key, value)

In [17]:
res = dict()

def evaluate_policy(args, env, agent, state_norm):
    num_episodes = 1
    rewards = []
    overloads = []
    intensities = []
    sdos = []
    fdos = []
    tdos = []
    ados = []

    for _ in range(num_episodes):
        s = env.reset()
        if args.use_state_norm:  # During the evaluating,update=False
            s = state_norm(s, update=False)
        s_noise = s
        done = False

        history_obs = []
        while not done:
            a = agent.evaluate(s_noise)  # We use the deterministic policy during the evaluating
            s_, r, done, info = env.step(np.array(a))
            # 此时的s_ 是真实的观测值，接下来对其进行噪声处理（注意深拷贝 浅拷贝）
            # 添加噪声
            if args.use_state_norm:
                s_ = state_norm(s_, update=False)
                
            if args.noise_relative_std != 0:
                s_noise_, history_obs = add_noise_to_state(s_, history_obs, args, args.noise_relative_std)
            else:
                s_noise_ = s_
            
            s, s_noise = s_, s_noise_

        ep_r, ep_overload, ep_intensity, ep_sdo, ep_fdo, ep_tdo, ep_ado = info.values()

        # 从env中输出此次感染情况：总新增感染人数，最大峰值
        total_new_I = sum(env.daily_new_I)
        daily_curr_I = np.sum(env.simRes[:, :, 2], axis=1)
        print(f"总新增感染人数：{total_new_I}，现存感染最大峰值：{max(daily_curr_I)}")
        rewards.append(ep_r)
        overloads.append(ep_overload)
        intensities.append(ep_intensity / env.period / env.ZONE_NUM)
        tdos.append(ep_tdo / env.ZONE_NUM)
        sdos.append(ep_sdo / env.period)
        fdos.append(ep_fdo / env.period)
        ados.append(ep_ado / env.period)

    result_dict = {
        'reward': rewards,
        'IOR': overloads,
        'ACI': intensities,
        'ATO': tdos,
        'ASO_adj': sdos,
        'ASO_mob': fdos,
        'ASO_adm': ados
    }

    return result_dict


def my_test(args):
    model_idx = args.model_idx
    eval_env = EpidemicModel(reward_mode=args.experiment_idx, city=args.city, R0=args.R0)
    args.zone_num = eval_env.ZONE_NUM
    agent = PPO_discrete(args)
    agent.load(model_idx)

    res_dict = evaluate_policy(args, eval_env, agent, args.use_state_norm)

    res[experiment_scenario[args.experiment_idx]] = res_dict

    eval_env.close()
    
    return res_dict


In [20]:
args.city = "sz"
args.R0 = 'high'

# 测试时加的噪声值
noise_relative_std = [0, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
# noise_relative_std = [0]
obs_noise_std = [0, 0.05, 0.1, 0.2, 0.3]
# obs_noise_std = [0]

args.experiment_idx = 4

args.model_idx = 100    
args.max_train_steps = int(1.2e6)

# 初始化存储结果的字典
results = {
    'reward': pd.DataFrame(0, index=noise_relative_std, columns=obs_noise_std),
    'IOR': pd.DataFrame(0, index=noise_relative_std, columns=obs_noise_std),
    'ACI': pd.DataFrame(0, index=noise_relative_std, columns=obs_noise_std)
}

for test_std in noise_relative_std:
    print(f" \n noise_std: {test_std:.2f}" + "=" * 50)
    args.noise_relative_std = test_std
    for train_std in obs_noise_std:
        print(f" \n obs_noise_std: {train_std:.2f}" + "-" * 50)
        args.obs_noise_std = train_std          
        if train_std != 0:
            args.model_idx = 70    
            args.max_train_steps = int(1.2e4 * 70)            
            args.use_obs_noise = True
            args.use_asymmetric = True
        else:
            args.model_idx = 100    
            args.max_train_steps = int(1.2e6)
            args.use_obs_noise = False
            args.use_asymmetric = False
        
        repeat = 5
        for _ in range(repeat):
            # 假设 my_test 返回一个字典，包含 'reward', 'IOR', 'ACI'
            output = my_test(args)
            
            # 计算每个列表的平均值，并累加
            avg_reward = sum(output['reward']) / len(output['reward'])
            avg_IOR = sum(output['IOR']) / len(output['IOR'])
            avg_ACI = sum(output['ACI']) / len(output['ACI'])
        
            results['reward'].at[test_std, train_std] += avg_reward / repeat
            results['IOR'].at[test_std, train_std] += avg_IOR / repeat
            results['ACI'].at[test_std, train_std] += avg_ACI / repeat

# 打印结果表格
print("Reward Table:")
print(results['reward'])
print("\nIOR Table:")
print(results['IOR'])
print("\nACI Table:")
print(results['ACI'])
    

 
 noise_std: 0.00==================================================
 
 obs_noise_std: 0.00--------------------------------------------------
总新增感染人数：16843342.3489808，现存感染最大峰值：3711629.214661751
总新增感染人数：16843342.3489808，现存感染最大峰值：3711629.214661751
总新增感染人数：16843342.3489808，现存感染最大峰值：3711629.214661751
总新增感染人数：16843342.3489808，现存感染最大峰值：3711629.214661751
总新增感染人数：16843342.3489808，现存感染最大峰值：3711629.214661751
 
 obs_noise_std: 0.05--------------------------------------------------
总新增感染人数：16888724.90601324，现存感染最大峰值：3921600.0573568987
总新增感染人数：16888724.90601324，现存感染最大峰值：3921600.0573568987
总新增感染人数：16888724.90601324，现存感染最大峰值：3921600.0573568987
总新增感染人数：16888724.90601324，现存感染最大峰值：3921600.0573568987
总新增感染人数：16888724.90601324，现存感染最大峰值：3921600.0573568987
 
 obs_noise_std: 0.10--------------------------------------------------
总新增感染人数：16770618.284697592，现存感染最大峰值：3837556.3493974414
总新增感染人数：16770618.284697592，现存感染最大峰值：3837556.3493974414
总新增感染人数：16770618.284697592，现存感染最大峰值：3837556.3493974414
总新增感染人数：16770618.

## action噪声测试

> 对action进行干扰，是直接影响到智能体的执行，对结果的影响是直观的，可能是缺乏研究价值的。



In [16]:
from config import args
from config import experiment_scenario

res = dict()
WINDOW_SIZE = 7
history_obs = []

def evaluate_policy(args, env, agent, state_norm):
    num_episodes = 1
    rewards = []
    overloads = []
    intensities = []
    sdos = []
    fdos = []
    tdos = []
    ados = []

    for _ in range(num_episodes):
        s = env.reset()
        if args.use_state_norm:  # During the evaluating,update=False
            s = state_norm(s, update=False)
        done = False

        while not done:

            a = agent.evaluate(s)  # We use the deterministic policy during the evaluating
            # TODO: 添加行动的噪声
            true_data = np.array(a)
            noise_std = args.noise_relative_std * (true_data.max() - true_data.min())
            noise = np.random.normal(0, noise_std, size=true_data.shape)
            a = true_data - np.abs(noise)
            s_, r, done, info = env.step(np.array(a))            
            
            if args.use_state_norm:
                s_ = state_norm(s_, update=False)

            s = s_

        ep_r, ep_overload, ep_intensity, ep_sdo, ep_fdo, ep_tdo, ep_ado = info.values()

        # env.render()
        # 从env中输出此次感染情况：总新增感染人数，最大峰值
        total_new_I = sum(env.daily_new_I)
        daily_curr_I = np.sum(env.simRes[:, :, 2], axis=1)
        print(f"总新增感染人数：{total_new_I}，现存感染最大峰值：{max(daily_curr_I)}")
        rewards.append(ep_r)
        overloads.append(ep_overload)
        intensities.append(ep_intensity / env.period / env.ZONE_NUM)
        tdos.append(ep_tdo / env.ZONE_NUM)
        sdos.append(ep_sdo / env.period)
        fdos.append(ep_fdo / env.period)
        ados.append(ep_ado / env.period)

    result_dict = {
        'reward': rewards,
        'IOR': overloads,
        'ACI': intensities,
        'ATO': tdos,
        'ASO_adj': sdos,
        'ASO_mob': fdos,
        'ASO_adm': ados
    }

    return result_dict


def my_test(args):
    model_idx = args.model_idx
    eval_env = EpidemicModel(reward_mode=args.experiment_idx, city=args.city, R0=args.R0)
    args.zone_num = eval_env.ZONE_NUM
    agent = PPO_discrete(args)
    agent.load(model_idx)

    res_dict = evaluate_policy(args, eval_env, agent, args.use_state_norm)

    res[experiment_scenario[args.experiment_idx]] = res_dict

    eval_env.close()
    
    
args.city = "sz"
args.R0 = 'high'

noise_relative_std = [0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]

args.experiment_idx = 4
args.model_idx = 100
args.max_train_steps = int(1.2e6)
for std in noise_relative_std:
    print(f" \n noise_std: {std:.2f}")
    args.noise_relative_std = std
    my_test(args)
    print(res)


 
 noise_std: 0.00
总新增感染人数：16680923.64805069，现存感染最大峰值：3784586.207716196
{'basic': {'reward': [-67.70270270270268], 'IOR': [-0.05385344807095104], 'ACI': [0.5641891891891891], 'ATO': [5.418918918918919], 'ASO_adj': [0.44710341073854465], 'ASO_mob': [0.39989016492135365], 'ASO_adm': [0.5235924550227546]}}
 
 noise_std: 0.05
总新增感染人数：16761110.640928043，现存感染最大峰值：4012362.9540791903
{'basic': {'reward': [-58.83402145976036], 'IOR': [0.0030907385197975674], 'ACI': [0.48865010011909793], 'ATO': [11.54007311088491], 'ASO_adj': [2.0526656816611726], 'ASO_mob': [1.6833333333333333], 'ASO_adm': [2.5135783918743595]}}
 
 noise_std: 0.10
总新增感染人数：16830200.49727155，现存感染最大峰值：4234331.301852586
{'basic': {'reward': [-66.57962682938904], 'IOR': [0.058582825463146436], 'ACI': [0.41629251609703893], 'ATO': [17.717674792752323], 'ASO_adj': [2.072989104251877], 'ASO_mob': [1.7], 'ASO_adm': [2.5384653066453926]}}
 
 noise_std: 0.20
总新增感染人数：16959671.68530022，现存感染最大峰值：4690916.871679634
{'basic': {'reward': [-105.

## ODE有关参数变化测试

> 与 ODE（环境的一部分） 有关参数有（感染者移动比例、beta、潜伏期、恢复期）：
> - Pm:
> - beta:
> - gamma:
> - sigma: 

In [17]:
WINDOW_SIZE = args.WINDOW_SIZE

def evaluate_policy(args, env, agent, state_norm):
    num_episodes = 1
    rewards = []
    overloads = []
    intensities = []
    sdos = []
    fdos = []
    tdos = []
    ados = []

    for _ in range(num_episodes):
        s = env.reset()
        if args.use_state_norm:  # During the evaluating,update=False
            s = state_norm(s, update=False)
        s_noise = s
        done = False

        history_obs = []
        while not done:
            a = agent.evaluate(s_noise)  # We use the deterministic policy during the evaluating
            s_, r, done, info = env.step(np.array(a))
            # 此时的s_ 是真实的观测值，接下来对其进行噪声处理（注意深拷贝 浅拷贝）
            # 添加噪声
            if args.use_state_norm:
                s_ = state_norm(s_, update=False)
            
            if args.noise_relative_std != 0:
                s_noise_, history_obs = add_noise_to_state(s_, history_obs, args, args.noise_relative_std)
            else:
                s_noise_ = s_
            
            s, s_noise = s_, s_noise_

        ep_r, ep_overload, ep_intensity, ep_sdo, ep_fdo, ep_tdo, ep_ado = info.values()

        # env.render()
        # 从env中输出此次感染情况：总新增感染人数，最大峰值
        total_new_I = sum(env.daily_new_I)
        daily_curr_I = np.sum(env.simRes[:, :, 2], axis=1)
        print(f"总新增感染人数：{total_new_I}，现存感染最大峰值：{max(daily_curr_I)}")
        rewards.append(ep_r)
        overloads.append(ep_overload)
        intensities.append(ep_intensity / env.period / env.ZONE_NUM)
        tdos.append(ep_tdo / env.ZONE_NUM)
        sdos.append(ep_sdo / env.period)
        fdos.append(ep_fdo / env.period)
        ados.append(ep_ado / env.period)

    result_dict = {
        'reward': rewards,
        'IOR': overloads,
        'ACI': intensities,
        'ATO': tdos,
        'ASO_adj': sdos,
        'ASO_mob': fdos,
        'ASO_adm': ados
    }

    return result_dict


def my_test(args):
    model_idx = args.model_idx
    eval_env = EpidemicModel(reward_mode=args.experiment_idx, city=args.city, R0=args.R0)
    eval_env.set_init_params(args)
    args.zone_num = eval_env.ZONE_NUM
    agent = PPO_discrete(args)
    agent.load(model_idx)

    res_dict = evaluate_policy(args, eval_env, agent, args.use_state_norm)
    
    eval_env.close()
    
    return res_dict


In [18]:
args.city = "sz"
args.R0 = 'high'
args.experiment_idx = 4
args.model_idx = 100
args.max_train_steps = int(1.2e6)

ODE_Pm = args.ODE_Pm
ODE_beta = args.ODE_beta
ODE_gamma = args.ODE_gamma
ODE_sigma = args.ODE_sigma

# 参数列表
param_names = ['ODE_Pm', 'ODE_beta', 'ODE_gamma', 'ODE_sigma']
original_values = [ODE_Pm, ODE_beta, ODE_gamma, ODE_sigma]

# 对ODE的参数分别 +- 0.05, 0.1, 0.2, 0.3, 0.4, 0.5
ODE_relative_std = [0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]

# 结果记录到一个字典中
ODE_params_noise_res = {}

for param_name, original_value in zip(param_names, original_values):
    for std in ODE_relative_std:
        # 正向调整
        setattr(args, param_name, original_value * (1 + std))
        print(f"Adjusting {param_name} to {original_value * (1 + std)}")
        res_dict_plus = my_test(args)
        ODE_params_noise_res[f"{param_name} +{std}"] = [res_dict_plus[key] for key in ['reward', 'IOR', 'ACI']]
        
        # 负向调整
        setattr(args, param_name, original_value * (1 - std))
        print(f"Adjusting {param_name} to {original_value * (1 - std)}")
        res_dict_minus = my_test(args)
        ODE_params_noise_res[f"{param_name} -{std}"] = [res_dict_minus[key] for key in ['reward', 'IOR', 'ACI']]
        
        # 恢复原始值
        setattr(args, param_name, original_value)

# 输出结果
# print("ODE Parameters Noise Results:")
# for key, results in ODE_params_noise_res.items():
#     print(f"Adjustment: {key}, Results: {results}")
    

Adjusting ODE_Pm to 0.4
总新增感染人数：16571542.43357216，现存感染最大峰值：4033820.2854347816
Adjusting ODE_Pm to 0.4
总新增感染人数：16631845.084985249，现存感染最大峰值：4023210.21907509
Adjusting ODE_Pm to 0.42000000000000004
总新增感染人数：16705400.07631505，现存感染最大峰值：3998092.3916292377
Adjusting ODE_Pm to 0.38
总新增感染人数：16510611.74008596，现存感染最大峰值：3907356.953970193
Adjusting ODE_Pm to 0.44000000000000006
总新增感染人数：16742491.807065343，现存感染最大峰值：4152890.763381879
Adjusting ODE_Pm to 0.36000000000000004
总新增感染人数：16397984.276151229，现存感染最大峰值：3668277.7667209273
Adjusting ODE_Pm to 0.48
总新增感染人数：16778370.588279556，现存感染最大峰值：4215869.265942336
Adjusting ODE_Pm to 0.32000000000000006
总新增感染人数：16283074.597598204，现存感染最大峰值：3526636.2529221764
Adjusting ODE_Pm to 0.52
总新增感染人数：17027788.744478095，现存感染最大峰值：4413087.4523968715
Adjusting ODE_Pm to 0.27999999999999997
总新增感染人数：16016899.384200528，现存感染最大峰值：3294295.181618977
Adjusting ODE_Pm to 0.5599999999999999
总新增感染人数：17099564.40321183，现存感染最大峰值：4616033.93970607
Adjusting ODE_Pm to 0.24
总新增感染人数：15420071.111

In [20]:
# 以表格形式输出易于观察对比的结果
relative_std = ODE_relative_std

# 生成包含正负符号的行索引
index_names = [f"+{std}" for std in relative_std] + [f"-{std}" for std in relative_std]
index_names.sort(key=lambda x: float(x.strip('+').strip('-')), reverse=False)

# 创建三个空的 DataFrame
reward_df = pd.DataFrame(index=index_names, columns=param_names)
ior_df = pd.DataFrame(index=index_names, columns=param_names)
aci_df = pd.DataFrame(index=index_names, columns=param_names)

# 填充数据到各个 DataFrame 中
for param in param_names:
    for std in relative_std:
        # 正向调整的结果
        pos_key = f"{param} +{std}"
        neg_key = f"{param} -{std}"
        
        # 为正向和负向调整填充数据
        reward_df.at[f"+{std}", param] = ODE_params_noise_res[pos_key][0][0]
        reward_df.at[f"-{std}", param] = ODE_params_noise_res[neg_key][0][0]
        
        ior_df.at[f"+{std}", param] = ODE_params_noise_res[pos_key][1][0]
        ior_df.at[f"-{std}", param] = ODE_params_noise_res[neg_key][1][0]
        
        aci_df.at[f"+{std}", param] = ODE_params_noise_res[pos_key][2][0]
        aci_df.at[f"-{std}", param] = ODE_params_noise_res[neg_key][2][0]

# 输出三个 DataFrame
print("Reward Table:")
print(reward_df)
print("\nIOR Table:")
print(ior_df)
print("\nACI Table:")
print(aci_df)

Reward Table:
           ODE_Pm    ODE_beta   ODE_gamma   ODE_sigma
+0     -77.936118  -83.081081  -80.486486       -86.5
-0     -78.389999  -77.013514  -82.891892       -82.0
+0.05  -77.364865  -90.767612  -82.837838  -79.054054
-0.05  -81.364865  -82.054054  -82.342919  -83.891892
+0.1   -87.663429 -124.251862  -76.797297  -76.918919
-0.1   -84.986486  -99.662162 -106.885971  -85.409128
+0.2   -97.039407 -186.335021  -80.959459  -87.189189
-0.2   -80.337838  -97.513514 -188.063913 -100.915013
+0.3    -113.2014 -261.845137   -77.27027  -88.608108
-0.3   -85.594595 -119.567568 -338.145462  -122.06406
+0.4  -138.459252 -326.123248  -78.216216  -91.378378
-0.4   -93.054054 -125.648649 -529.207437  -146.99174
+0.5  -155.516462 -365.112051  -78.162162  -93.837838
-0.5   -97.364865 -146.932432 -816.237202 -151.045858

IOR Table:
         ODE_Pm  ODE_beta ODE_gamma ODE_sigma
+0     0.008455 -0.031127 -0.029947 -0.027013
-0     0.005803  -0.01117 -0.030833 -0.029158
+0.05 -0.000477  0.038722 

## 时间统计

In [22]:
import pstats

# 创建 Stats 对象
stats = pstats.Stats('C:/Users/21268/AppData/Local/JetBrains/PyCharm2024.2/snapshots/DRL_EPC_STO/DRL_EPC_STO.pstat')

# 展示所有统计信息
# stats.print_stats()

# 根据需要可以过滤和排序信息
stats.sort_stats('cumulative').print_stats(10)  # 展示累计时间最长的前10条记录


Tue Aug 27 15:25:13 2024    C:/Users/21268/AppData/Local/JetBrains/PyCharm2024.2/snapshots/DRL_EPC_STO/DRL_EPC_STO.pstat

         1448888794 function calls (1377571491 primitive calls) in 1708.152 seconds

   Ordered by: cumulative time
   List reduced from 5628 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    97833   12.539    0.000 1004.890    0.010 D:\E\code\reinforce_project\DRL_EPC_STO\ppo_discrete.py:147(choose_action)
    98793   10.937    0.000  520.698    0.005 D:\E\code\reinforce_project\DRL_EPC_STO\meta_env.py:170(step)
   296378  106.608    0.000  499.248    0.002 D:\E\code\reinforce_project\DRL_EPC_STO\meta_env.py:97(spatio_entropy)
  7573530   44.189    0.000  390.800    0.000 D:\ProgramData\anaconda3\envs\myenv-DRL-EPC-STO-lx\lib\site-packages\torch\distributions\categorical.py:121(log_prob)
    97833    7.147    0.000  377.591    0.004 D:\E\code\reinforce_project\DRL_EPC_STO\ppo_discrete.py:158(<listcomp>)
93478